In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
print(train.head())  

test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')
print(test.head())

sample_submission = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/sample_submission.csv')
print(sample_submission.head())

In [ ]:
train.columns

In [ ]:
print("Все колонки: ", train.columns.tolist())
train.info()

Возьмём в качестве baseline линейную регрессию. На её основе хотя бы начнём отбирать признаки, с коэффициентом 0 убираем. Потом применим регуляризацию L1 и посмотрим, что останется

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

Линейная регрессия:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


feature_cols = [col for col in train.columns if col not in ['ID', 'target']]

X_train = train[feature_cols].values
y_train = train['target'].values
X_test = test[feature_cols].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = Lasso(alpha=0.001)
model.fit(X_train_sc, y_train)
preds = model.predict(X_test_sc)

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': preds
})
submission.to_csv('submission.csv', index=False)
print("Submission saved")
print(submission.head())


coefs = model.coef_

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': coefs
})

# Оставляем только те признаки, у которых коэффициент не равен нулю 
coef_df_nonzero = coef_df[coef_df['Coefficient'] != 0]

coef_df_nonzero['Abs_Coef'] = coef_df_nonzero['Coefficient'].abs()
coef_df_nonzero = coef_df_nonzero.sort_values(by='Abs_Coef', ascending=False)

top_coefs = coef_df_nonzero.head(20)
plt.figure(figsize=(12, 8))
colors = ['#d62728' if c > 0 else '#2ca02c' for c in top_coefs['Coefficient']]

plt.barh(top_coefs['Feature'], top_coefs['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Значение коэффициента (Lasso weight)', fontsize=12)
plt.ylabel('Признак (Feature)', fontsize=12)
plt.title('Топ-20 признаков (по коэффициентам Lasso)', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.gca().invert_yaxis() 
plt.tight_layout()
plt.show()

print(f"Всего признаков: {len(feature_cols)}")
print(f"Lasso оставил (ненулевых): {len(coef_df_nonzero)}")

Теперь применим PCA для снижения размерностиpca_full = PCA()


In [ ]:
feature_cols_new = coef_df_nonzero['Feature'].tolist()
X_train_after_lin_reg = train[feature_cols_new].values
y_train = train['target'].values
X_test_after_lin_reg = test[feature_cols_new].values

scaler = StandardScaler()
X_train_sc_after_lin_reg = scaler.fit_transform(X_train_after_lin_reg)
X_test_sc_after_lin_reg = scaler.transform(X_test_after_lin_reg)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

def plot_pca_with_threshold(pca_model, title):
    cumsum = np.cumsum(pca_model.explained_variance_ratio_)
    idx_95 = np.where(cumsum >= 0.95)[0][0]
    n_comps_to_95 = idx_95 + 1  # +1, потому что индексы начинаются с 0
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(cumsum) + 1), cumsum, 
             marker='o', linestyle='--', color='b', label='Кумулятивная дисперсия')
    
    plt.axhline(y=0.95, color='r', linestyle='-', label='95% дисперсии')
    
    plt.axvline(x=n_comps_to_95, color='g', linestyle='--', 
                label=f'Пересечение на {n_comps_to_95} компонентах')
    
    plt.plot(n_comps_to_95, 0.95, 'ro', markersize=8, label='Точка пересечения')
    plt.xlabel('Число компонент (n_components)')
    plt.ylabel('Объясненная дисперсия')
    plt.title(title)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    
    print(f"Для достижения 95% дисперсии достаточно {n_comps_to_95} компонент.")
    return n_comps_to_95

pca_full = PCA()
pca_full.fit(X_train_sc)
n_1 = plot_pca_with_threshold(pca_full, 'PCA на всех признаках (X_train_sc)')

if 'X_train_sc_after_lin_reg' in locals():
    pca_after = PCA()
    pca_after.fit(X_train_sc_after_lin_reg)
    n_2 = plot_pca_with_threshold(pca_after, 'PCA после отбора Lasso (X_train_sc_after_lin_reg)')
else:
    print("Переменная X_train_sc_after_lin_reg не найдена. Пропускаем второй график.")

PCA считает, что есть 1527 важных направлений дисперсии,учтём на будущее, что target можно прологарифмировать. Получим сжатые данные и пойдём в Random Forest

In [ ]:
from sklearn.decomposition import PCA
pca_final = PCA(n_components=1527, random_state=42)

X_train_pca_final = pca_final.fit_transform(X_train_sc_after_lin_reg)
X_test_pca_final = pca_final.transform(X_test_sc_after_lin_reg)

print(f"Было колонок: {X_train_sc_after_lin_reg.shape[1]}")
print(f"Стало колонок: {X_train_pca_final.shape[1]}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

y_full_train = train['target'].values
y_full_log = np.log1p(y_full_train)  # log1p(x) = log(x + 1)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_pca_final, y_full_log, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2, 
                            random_state=42, n_jobs=-1)
rf.fit(X_train_split, y_train_split)
y_pred_log = rf.predict(X_val_split)
rmse = np.sqrt(mean_squared_error(y_val_split, y_pred_log))
r2 = r2_score(y_val_split, y_pred_log)

print(f"RMSE (log): {rmse:.4f}")
print(f"R² (log): {r2:.4f}")

rf_final = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2, 
                                  random_state=42, n_jobs=-1)
rf_final.fit(X_train_pca_final, y_full_log)

preds_log = rf_final.predict(X_test_pca_final)
preds = np.expm1(preds_log)
preds = np.maximum(preds, 0)  

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': preds
})
submission.to_csv('submission_rf_final.csv', index=False)

In [ ]:
y_full_train = train['target'].values
y_full_log = np.log1p(y_full_train)  # log1p(x) = log(x + 1)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_pca_final, y_full_log, test_size=0.2, random_state=42
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

feature_importances = rf_final.feature_importances_
pca_feature_names = [f'PC_{i+1}' for i in range(len(feature_importances))]

imp_df = pd.DataFrame({
    'Feature': pca_feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

print("\nТоп-200 самых важных компонент (PCA)")
print(imp_df.head(200).to_string(index=False))

plt.figure(figsize=(12, 8))
top_n = 20
top_features = imp_df.head(top_n)

sns.barplot(x='Importance', y='Feature', data=top_features, palette='viridis')

plt.title(f'Топ-{top_n} важнейших признаков (PCA-компонент) в Random Forest', fontsize=14)
plt.xlabel('Важность (Feature Importance)', fontsize=12)
plt.ylabel('Главная компонента (PC)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)

for i, (_, row) in enumerate(top_features.iterrows()):
    plt.text(row['Importance'] + 0.001, i, f'{row["Importance"]:.4f}', 
             va='center', fontsize=9, color='black')

plt.tight_layout()
plt.show()

Попробуем оставить 200 признаков вместо 1527 и ещё раз запустить обучение на Random Forest

In [ ]:
X_train_pca_top200 = X_train_pca_final[:, :200]
X_test_pca_top200 = X_test_pca_final[:, :200]

print(f"Размер тренировочных данных: {X_train_pca_top200.shape}")
print(f"Размер тестовых данных: {X_test_pca_top200.shape}")

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_pca_top200, y_full_log, test_size=0.2, random_state=42
)


rf_top200 = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf_top200.fit(X_train_split, y_train_split)

y_pred_val_top200 = rf_top200.predict(X_val_split)

rmse_top200 = np.sqrt(mean_squared_error(y_val_split, y_pred_val_top200))
r2_top200 = r2_score(y_val_split, y_pred_val_top200)

print(f"RMSE (log): {rmse_top200:.4f}")
print(f"R² (log): {r2_top200:.4f}")

rf_final_top200 = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf_final_top200.fit(X_train_pca_top200, y_full_log)

preds_log_top200 = rf_final_top200.predict(X_test_pca_top200)

preds_top200 = np.expm1(preds_log_top200)
preds_top200 = np.maximum(preds_top200, 0)

submission_top200 = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_top200
})
submission_top200.to_csv('submission_rf_top200.csv', index=False)

print("\nСабмит с 200 компонентами сохранен!")

In [ ]:
X_train_pca_top150 = X_train_pca_final[:, :150]
X_test_pca_top150 = X_test_pca_final[:, :150]

print(f"Размер тренировочных данных: {X_train_pca_top150.shape}")
print(f"Размер тестовых данных: {X_test_pca_top150.shape}")

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_pca_top150, y_full_log, test_size=0.2, random_state=42
)


И да, на 200 признаках R^2 увеличился, вместо 0.2084 стал 0.2352. Если попробовать 150 признаков?

In [ ]:
X_train_pca_top150 = X_train_pca_final[:, :150]
X_test_pca_top150 = X_test_pca_final[:, :150]

print(f"Размер тренировочных данных: {X_train_pca_top150.shape}")
print(f"Размер тестовых данных: {X_test_pca_top150.shape}")

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_pca_top150, y_full_log, test_size=0.2, random_state=42
)


rf_top150 = RandomForestRegressor(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1)
rf_top150.fit(X_train_split, y_train_split)

y_pred_val_top150 = rf_top150.predict(X_val_split)

rmse_top150 = np.sqrt(mean_squared_error(y_val_split, y_pred_val_top150))
r2_top150 = r2_score(y_val_split, y_pred_val_top150)

print(f"RMSE (log): {rmse_top150:.4f}")
print(f"R² (log): {r2_top150:.4f}")

rf_final_top150 = RandomForestRegressor(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1)
rf_final_top150.fit(X_train_pca_top150, y_full_log)

preds_log_top150 = rf_final_top150.predict(X_test_pca_top150)

preds_top150 = np.expm1(preds_log_top150)
preds_top150 = np.maximum(preds_top150, 0)

submission_top150 = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_top150
})
submission_top150.to_csv('submission_rf_top150.csv', index=False)

print("\nСабмит с 150 компонентами сохранен!")

Если оставить 150 признаков, R^2 ещё немного вырос. С 0.2352 до 0.2362. Займёмся подбором параметров для Random Forest

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

n_trees_list = [1, 5, 10, 20, 50, 60, 70, 80, 90, 100, 150, 200, 300]
rmse_scores = []
r2_scores = []

last_rf = None

for n_trees in n_trees_list:
    if last_rf is None:
        rf_temp = RandomForestRegressor(n_estimators=n_trees, max_depth=20,
                                        random_state=42, n_jobs=-1)
    else:
        rf_temp = last_rf
        rf_temp.n_estimators = n_trees  

    rf_temp.fit(X_train_split, y_train_split)
    y_pred_val = rf_temp.predict(X_val_split)

    rmse_scores.append(np.sqrt(mean_squared_error(y_val_split, y_pred_val)))
    r2_scores.append(r2_score(y_val_split, y_pred_val))
    last_rf = rf_temp

# приводим обе метрики к шкале 0..1
rmse_norm = (rmse_scores - np.min(rmse_scores)) / (np.max(rmse_scores) - np.min(rmse_scores))
r2_norm = (r2_scores - np.min(r2_scores)) / (np.max(r2_scores) - np.min(r2_scores))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(n_trees_list, rmse_norm, color='tab:red', marker='o', 
        label='RMSE (норм.)', linewidth=2)
ax.plot(n_trees_list, r2_norm, color='tab:blue', marker='s', linestyle='--', 
        label='R² (норм.)', linewidth=2)

ax.set_xlabel('Число деревьев (n_estimators)', fontsize=12)
ax.set_ylabel('Нормализованное значение (0 = мин, 1 = макс)', fontsize=12)
ax.set_title('Random Forest: сравнение динамики RMSE и R² (нормализовано)', fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='center right', fontsize=10)

plt.tight_layout()
plt.show()

print("\nТаблица метрик по числу деревьев:")
for i, n in enumerate(n_trees_list):
    print(f"Trees={n:3d} | RMSE={rmse_scores[i]:.4f} | R²={r2_scores[i]:.4f}")

После 50 деревьев R^2 меняется на +0.006, 0.005, 0.003 на 100, 150, 200 деревьев соответственно. При более детальном анализе количества деревьев от 50 до 100 выяснилось, что оптимальное количество 80 

Trees= 70 | RMSE=1.4875 | R²=0.2310
Trees= 80 | RMSE=1.4845 | R²=0.2341
Trees= 90 | RMSE=1.4877 | R²=0.2308

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

print(f"Train shape: {X_train_split.shape}")
print(f"Val shape:   {X_val_split.shape}")

# Параметры XGBoost для регрессии
xgb_model = xgb.XGBRegressor(
    n_estimators=80,              
    max_depth=6,                  # стандартная глубина для XGB (обычно 4-6)
    learning_rate=0.1,            # шаг обучения (чем меньше, тем точнее, но медленнее)
    subsample=0.8,                # доля строк для каждого дерева (борьба с переобучением)
    colsample_bytree=0.8,         # доля признаков для каждого дерева
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=10,     # ранняя остановка, если валидация перестала улучшаться
    eval_metric='rmse'            # метрика для ранней остановки
)

xgb_model.fit(
    X_train_split, 
    y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    verbose=False
)

y_pred_xgb_val = xgb_model.predict(X_val_split)

rmse_xgb = np.sqrt(mean_squared_error(y_val_split, y_pred_xgb_val))
r2_xgb = r2_score(y_val_split, y_pred_xgb_val)

print(f"\nМетрики XGBoost (на валидации):")
print(f"RMSE (log): {rmse_xgb:.4f}")
print(f"R² (log):   {r2_xgb:.4f}")

xgb_final = xgb.XGBRegressor(
    n_estimators=80,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_final.fit(X_train_pca_top150, y_full_log)

preds_xgb_log = xgb_final.predict(X_test_pca_top150)
preds_xgb = np.expm1(preds_xgb_log)   
preds_xgb = np.maximum(preds_xgb, 0)  

submission_xgb = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_xgb
})
submission_xgb.to_csv('submission_xgb.csv', index=False)

print("\nСабмит XGBoost сохранен!")
print(submission_xgb.head())

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_xgb.csv',   
    message='XGBoost 80 trees, 150 PCA features',  
    competition='santander-value-prediction-challenge' 
)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

print(f"LightGBM Train shape: {X_train_split.shape}")
print(f"LightGBM Val shape:   {X_val_split.shape}")

lgb_model = lgb.LGBMRegressor(
    n_estimators=80,               # оптимальное число деревьев
    max_depth=6,                   # глубина дерева
    learning_rate=0.1,             # шаг обучения
    subsample=0.8,                 # доля строк (bagging_fraction)
    colsample_bytree=0.8,          # доля фич (feature_fraction)
    random_state=42,
    n_jobs=-1,
    verbose=-1,                    # отключить вывод в консоль
    early_stopping_rounds=10       # ранняя остановка для валидации
)

lgb_model.fit(
    X_train_split, 
    y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    eval_metric='rmse'
)

y_pred_lgb_val = lgb_model.predict(X_val_split)

rmse_lgb = np.sqrt(mean_squared_error(y_val_split, y_pred_lgb_val))
r2_lgb = r2_score(y_val_split, y_pred_lgb_val)

print(f"\nМетрики LightGBM (на валидации):")
print(f"RMSE (log): {rmse_lgb:.4f}")
print(f"R² (log):   {r2_lgb:.4f}")

lgb_final = lgb.LGBMRegressor(
    n_estimators=80,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_final.fit(X_train_pca_top150, y_full_log)

preds_lgb_log = lgb_final.predict(X_test_pca_top150)
preds_lgb = np.expm1(preds_lgb_log)
preds_lgb = np.maximum(preds_lgb, 0)

submission_lgb = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_lgb
})
submission_lgb.to_csv('submission_lgb.csv', index=False)

print("\nСабмит LightGBM сохранен!")
print(submission_lgb.head())

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_lgb.csv',   
    message='LightGBM 80 trees, 150 PCA features',  
    competition='santander-value-prediction-challenge' 
)

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

print(f"CatBoost Train shape: {X_train_split.shape}")
print(f"CatBoost Val shape:   {X_val_split.shape}")

cat_model = CatBoostRegressor(
    iterations=80,                # аналог n_estimators
    depth=6,                      # глубина дерева
    learning_rate=0.1,            # шаг обучения
    subsample=0.8,                # доля строк для каждого дерева (борьба с переобучением)
    random_seed=42,
    verbose=False,                # отключаем подробный вывод в консоль
    early_stopping_rounds=10      # ранняя остановка (сработает при передаче eval_set)
)

cat_model.fit(
    X_train_split, 
    y_train_split,
    eval_set=(X_val_split, y_val_split),
    use_best_model=True           # сохраняет лучшую модель по валидации
)

y_pred_cat_val = cat_model.predict(X_val_split)

rmse_cat = np.sqrt(mean_squared_error(y_val_split, y_pred_cat_val))
r2_cat = r2_score(y_val_split, y_pred_cat_val)

print(f"\nМетрики CatBoost (на валидации):")
print(f"RMSE (log): {rmse_cat:.4f}")
print(f"R² (log):   {r2_cat:.4f}")

cat_final = CatBoostRegressor(
    iterations=80,
    depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_seed=42,
    verbose=False
)
cat_final.fit(X_train_pca_top150, y_full_log)

preds_cat_log = cat_final.predict(X_test_pca_top150)

preds_cat = np.expm1(preds_cat_log)
preds_cat = np.maximum(preds_cat, 0)

submission_cat = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_cat
})
submission_cat.to_csv('submission_cat.csv', index=False)

print("\nСабмит CatBoost сохранен")
print(submission_cat.head())

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_cat.csv',   
    message='CatBoost 80 trees, 150 PCA features',  
    competition='santander-value-prediction-challenge' 
)

Итого: 

XGBoost: 

RMSE (log): 1.4784
R² (log):   0.2404

LightGBM: 

RMSE (log): 1.4811
R² (log):   0.2376


CatBoost: 

RMSE (log): 1.4958
R² (log):   0.2224

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

# depth: от 4 до 10 (с шагом 2, чтобы не перебирать все подряд)
# learning_rate: от 0.01 до 0.2 
depth_list = [4, 6, 8, 10]
lr_list = [0.01, 0.05, 0.1, 0.15, 0.2]

results = []

print("Начинаем перебор параметров CatBoost")

for depth in depth_list:
    for lr in lr_list:
        model = CatBoostRegressor(
            iterations=100,          
            depth=depth,
            learning_rate=lr,
            subsample=0.8,
            random_seed=42,
            verbose=False,
            early_stopping_rounds=10
        )
        
        model.fit(
            X_train_split, 
            y_train_split,
            eval_set=(X_val_split, y_val_split),
            verbose=False
        )
        
    
        y_pred = model.predict(X_val_split)
        rmse = np.sqrt(mean_squared_error(y_val_split, y_pred))
        
    
        results.append({
            'depth': depth,
            'learning_rate': lr,
            'rmse_val': rmse
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values('rmse_val')

print("\nТОП-5 лучших комбинаций параметров:")
print(results_df.head(5).round(4))

pivot_table = results_df.pivot(index='depth', columns='learning_rate', values='rmse_val')

plt.figure(figsize=(10, 6))
plt.imshow(pivot_table, cmap='RdYlGn_r', aspect='auto')
plt.colorbar(label='RMSE (log) — чем меньше, тем лучше')
plt.xticks(ticks=np.arange(len(lr_list)), labels=lr_list)
plt.yticks(ticks=np.arange(len(depth_list)), labels=depth_list)
plt.xlabel('learning_rate')
plt.ylabel('depth')
plt.title('CatBoost: RMSE на валидации (перебор параметров)')

for i in range(len(depth_list)):
    for j in range(len(lr_list)):
        plt.text(j, i, f"{pivot_table.iloc[i, j]:.4f}", 
                 ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

Применим ещё раз CatBoost с подобранными оптимальными гиперпараметрами:

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

print(f"CatBoost Train shape: {X_train_split.shape}")
print(f"CatBoost Val shape:   {X_val_split.shape}")

cat_model = CatBoostRegressor(
    iterations=80,                # аналог n_estimators
    depth=4,                      # глубина дерева
    learning_rate=0.15,            # шаг обучения
    subsample=0.8,                # доля строк для каждого дерева (борьба с переобучением)
    random_seed=42,
    verbose=False,                # отключаем подробный вывод в консоль
    early_stopping_rounds=10      # ранняя остановка (сработает при передаче eval_set)
)

cat_model.fit(
    X_train_split, 
    y_train_split,
    eval_set=(X_val_split, y_val_split),
    use_best_model=True           # сохраняет лучшую модель по валидации
)

y_pred_cat_val = cat_model.predict(X_val_split)

rmse_cat = np.sqrt(mean_squared_error(y_val_split, y_pred_cat_val))
r2_cat = r2_score(y_val_split, y_pred_cat_val)

print(f"\nМетрики CatBoost (на валидации):")
print(f"RMSE (log): {rmse_cat:.4f}")
print(f"R² (log):   {r2_cat:.4f}")

cat_final = CatBoostRegressor(
    iterations=80,
    depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_seed=42,
    verbose=False
)
cat_final.fit(X_train_pca_top150, y_full_log)

preds_cat_log = cat_final.predict(X_test_pca_top150)

preds_cat = np.expm1(preds_cat_log)
preds_cat = np.maximum(preds_cat, 0)

submission_cat = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_cat
})
submission_cat.to_csv('submission_cat_1.csv', index=False)

print("\nСабмит CatBoost сохранен")
print(submission_cat.head())

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_cat_1.csv',   
    message='CatBoost 80 trees, 150 PCA features, depth 4, lr 0.15',  
    competition='santander-value-prediction-challenge' 
)

Улучшений подобранные глубина и lr не дали при отправке. Попробуем кросс-валидацию для CatBoost

In [ ]:
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []
r2_scores = []

print(f"Запуск 5-Fold кросс-валидацию на {X_train_pca_top150.shape[0]} объектах\n")

fold = 1
for train_idx, val_idx in kf.split(X_train_pca_top150):
    X_train_fold = X_train_pca_top150[train_idx]
    X_val_fold = X_train_pca_top150[val_idx]
    y_train_fold = y_full_log[train_idx]
    y_val_fold = y_full_log[val_idx]
    
    model = CatBoostRegressor(
        iterations=80,
        depth=4,
        learning_rate=0.15,
        subsample=0.8,
        random_seed=42,
        verbose=False
    )
    model.fit(X_train_fold, y_train_fold, verbose=False)
    
    y_pred_fold = model.predict(X_val_fold)
    
    rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
    r2 = r2_score(y_val_fold, y_pred_fold)
    
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    
    print(f"Fold {fold}: RMSE = {rmse:.4f}, R² = {r2:.4f}")
    fold += 1

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)

print("\n" + "="*50)
print("Метрики 5-FOLD CV:")
print(f"RMSE (лог) : {mean_rmse:.4f} ± {std_rmse:.4f}")
print(f"R²   (лог) : {mean_r2:.4f} ± {std_r2:.4f}")
print("="*50)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
folds = range(1, 6)

ax.plot(folds, rmse_scores, marker='o', label='RMSE', color='tab:red')
ax.plot(folds, r2_scores, marker='s', label='R²', color='tab:blue', linestyle='--')

ax.set_xlabel('Номер фолда (Fold)')
ax.set_ylabel('Значение метрики')
ax.set_title('CatBoost (5-Fold CV): стабильность метрик по фолдам')
ax.set_xticks(folds)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np

param_grid = {
    'iterations': [80, 150, 300, 500],
    'learning_rate': [0.03, 0.05, 0.08, 0.1, 0.15],
    'depth': [4, 6, 8]
}

catboost_base = CatBoostRegressor(subsample=0.8, random_seed=42, verbose=False)

grid_search = GridSearchCV(
    estimator=catboost_base,
    param_grid=param_grid,
    cv=5,                     # 5-Fold CV
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,                # использовать все ядра
    verbose=1                 # показывать прогресс
)

# 4. Запускаем поиск на ваших данных
grid_search.fit(X_train_pca_top150, y_full_log)

# 5. Выводим результаты
print("\nЛучшая комбинация параметров: :")
print(grid_search.best_params_)
print(f"Лучший CV RMSE (лог): {-grid_search.best_score_:.4f}")

Как показал GridSearch, лучшие параметры для CatBoost: depth=8, iterations=500, learning_rate=0.05. Повторим обучение CatBoost с уже оптимальными параметрами

In [ ]:
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor
from sklearn.metrics import make_scorer, mean_squared_error


best_params = {'depth': 8, 'iterations': 500, 'learning_rate': 0.05}

cat_best = CatBoostRegressor(
    iterations=best_params['iterations'],
    depth=best_params['depth'],
    learning_rate=best_params['learning_rate'],
    subsample=0.8,
    random_seed=42,
    verbose=False
)

cat_best.fit(X_train_pca_top150, y_full_log)
preds_log = cat_best.predict(X_test_pca_top150)
preds = np.expm1(preds_log)
preds = np.maximum(preds, 0)

submission_cv = pd.DataFrame({
    'ID': test['ID'],
    'target': preds
})
submission_cv.to_csv('submission_cv_best.csv', index=False)

print("Сабмит с параметрами от GridSearchCV сохранен")
print(submission_cv.head())

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_cv_best.csv',   
    message='CatBoost 80 trees, 150 PCA features, depth 8, lr 0.05',  
    competition='santander-value-prediction-challenge' 
)

CatBoost с параметрами, подобранными через GridSearch оказался хуже, чем просто CatBoost. Попробуем блендинг

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np

cat_final = CatBoostRegressor(
    iterations=80, depth=4, learning_rate=0.15, 
    subsample=0.8, random_seed=42, verbose=False
)
cat_final.fit(X_train_pca_top150, y_full_log)
preds_cat = np.expm1(cat_final.predict(X_test_pca_top150))
preds_cat = np.maximum(preds_cat, 0)

lgb_final = LGBMRegressor(
    n_estimators=80, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
)
lgb_final.fit(X_train_pca_top150, y_full_log)
preds_lgb = np.expm1(lgb_final.predict(X_test_pca_top150))
preds_lgb = np.maximum(preds_lgb, 0)

weights = {'catboost': 0.7, 'lightgbm': 0.3}
preds_blend = preds_cat * weights['catboost'] + preds_lgb * weights['lightgbm']
preds_blend = np.maximum(preds_blend, 0)

submission_blend = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_blend
})
submission_blend.to_csv('submission_blend_70_30.csv', index=False)
print("Блендинг CatBoost (0.7) + LightGBM (0.3) сохранен!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_blend_70_30.csv',   
    message='CatBoost (0.7) + LightGBM (0.3)',  
    competition='santander-value-prediction-challenge' 
)

Блендинг пока показал лучший результат (1.63640). Попробуем стекинг

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

estimators = [
    ('cat', CatBoostRegressor(iterations=80, depth=4, learning_rate=0.15, 
                              subsample=0.8, random_seed=42, verbose=False)),
    ('lgb', LGBMRegressor(n_estimators=80, max_depth=6, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)),
    ('xgb', XGBRegressor(n_estimators=80, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1))
]

stack_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stack_reg.fit(X_train_pca_top150, y_full_log)
preds_stack_log = stack_reg.predict(X_test_pca_top150)

preds_stack = np.expm1(preds_stack_log)
preds_stack = np.maximum(preds_stack, 0)

# 6. Сабмит
submission_stack = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_stack
})
submission_stack.to_csv('submission_stack.csv', index=False)
print("Стекинг (CatBoost + LightGBM + XGBoost) сохранен!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_stack.csv',   
    message='Stacking CatBoost, LightGBM, XGBoost',  
    competition='santander-value-prediction-challenge' 
)

Стекинг оказался хуже, чем блендинг (1.64410 против 1.63640). Видимо, в стекинге базовые модели оказались слишком похожи и только усилили ошибку, взвешенное усреднение двух моделей (CatBoost + LightGBM) оказалось эффективнее. Попробуем поэкспериментировать с пропорциями в блендинге.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_cat = np.zeros(len(y_full_log))
oof_lgb = np.zeros(len(y_full_log))

for train_idx, val_idx in kf.split(X_train_pca_top150):
    X_tr, X_val = X_train_pca_top150[train_idx], X_train_pca_top150[val_idx]
    y_tr, y_val = y_full_log[train_idx], y_full_log[val_idx]

    cat_fold = CatBoostRegressor(iterations=80, depth=4, learning_rate=0.15, 
                                  subsample=0.8, random_seed=42, verbose=False)
    cat_fold.fit(X_tr, y_tr)
    oof_cat[val_idx] = cat_fold.predict(X_val)

    lgb_fold = LGBMRegressor(n_estimators=80, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
    lgb_fold.fit(X_tr, y_tr)
    oof_lgb[val_idx] = lgb_fold.predict(X_val)

best_score = float('inf')
best_w_cat = 0.7

print("Поиск лучших весов")
for w_cat in np.arange(0.50, 0.96, 0.01):
    w_lgb = 1.0 - w_cat 
    blend_oof = oof_cat * w_cat + oof_lgb * w_lgb
    rmse_oof = np.sqrt(mean_squared_error(y_full_log, blend_oof))
    
    if rmse_oof < best_score:
        best_score = rmse_oof
        best_w_cat = w_cat

best_w_lgb = 1.0 - best_w_cat
print(f"Лучшие веса: CatBoost = {best_w_cat:.2f}, LightGBM = {best_w_lgb:.2f}")
print(f"OOF RMSE (лог): {best_score:.4f}")

Подбор соотношений показал, что лучше всего 50 на 50. Проверим это ниже.

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np

cat_final = CatBoostRegressor(
    iterations=80, depth=4, learning_rate=0.15, 
    subsample=0.8, random_seed=42, verbose=False
)
cat_final.fit(X_train_pca_top150, y_full_log)
preds_cat = np.expm1(cat_final.predict(X_test_pca_top150))
preds_cat = np.maximum(preds_cat, 0)

lgb_final = LGBMRegressor(
    n_estimators=80, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
)
lgb_final.fit(X_train_pca_top150, y_full_log)
preds_lgb = np.expm1(lgb_final.predict(X_test_pca_top150))
preds_lgb = np.maximum(preds_lgb, 0)

weights = {'catboost': 0.5, 'lightgbm': 0.5}
preds_blend = preds_cat * weights['catboost'] + preds_lgb * weights['lightgbm']
preds_blend = np.maximum(preds_blend, 0)

submission_blend = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_blend
})
submission_blend.to_csv('submission_blend_50_50.csv', index=False)
print("Блендинг CatBoost (0.5) + LightGBM (0.5) сохранен!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_blend_50_50.csv',   
    message='CatBoost (0.5) + LightGBM (0.5)',  
    competition='santander-value-prediction-challenge' 
)

Действительно, стало лучше. 
submission_blend_50_50.csv: 1.63552 

submission_blend_70_30.csv: 1.63640 

Надо войти в топ-45%, пока удалось войти в топ-77%. Будем улучшать подбор гиперпараметров



In [ ]:
# берем топ-10 самых важных PCA-компонент (по feature importance из предыдущего RF)
X_pca_top10 = X_train_pca_top150[:, :10]
X_test_pca_top10 = X_test_pca_top150[:, :10]

# полиномиальные взаимодействия между ними 
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_poly = poly.fit_transform(X_pca_top10)
X_test_poly = poly.transform(X_test_pca_top10)

# склеиваем с основными 150 компонентами
import numpy as np
X_train_final = np.hstack((X_train_pca_top150, X_poly))
X_test_final = np.hstack((X_test_pca_top150, X_test_poly))

print(f"Новый размер тренировочных данных: {X_train_final.shape}")

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

# CatBoost
cat_final = CatBoostRegressor(
    iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=5.0,
    subsample=0.8, random_seed=42, verbose=False
)
cat_final.fit(X_train_final, y_full_log)
preds_cat_log = cat_final.predict(X_test_final)

# LightGBM
lgb_final = LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=31,
    min_child_samples=20, reg_lambda=10.0, subsample=0.8,
    colsample_bytree=0.8, random_state=42, verbose=-1
)
lgb_final.fit(X_train_final, y_full_log)
preds_lgb_log = lgb_final.predict(X_test_final)

# XGBoost
xgb_final = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
    colsample_bytree=0.8, reg_lambda=10.0, reg_alpha=0.1,
    random_state=42, n_jobs=-1
)
xgb_final.fit(X_train_final, y_full_log)
preds_xgb_log = xgb_final.predict(X_test_final)

# Ridge (на исходных масштабированных фичах)
ridge_model = Ridge(alpha=5.0)
ridge_model.fit(X_train_sc_after_lin_reg, y_full_log)
preds_ridge_log = ridge_model.predict(X_test_sc_after_lin_reg)

def safe_expm1(log_preds, max_log=20):
    log_preds = np.clip(log_preds, -max_log, max_log)
    preds = np.expm1(log_preds)
    preds = np.nan_to_num(preds, nan=0.0, posinf=0.0, neginf=0.0)
    return preds

preds_cat = safe_expm1(preds_cat_log)
preds_lgb = safe_expm1(preds_lgb_log)
preds_xgb = safe_expm1(preds_xgb_log)
preds_ridge = safe_expm1(preds_ridge_log)

weights = {
    'catboost': 0.50,
    'lightgbm': 0.30,
    'xgboost': 0.10,
    'ridge': 0.10
}

preds_blend = (
    preds_cat * weights['catboost'] +
    preds_lgb * weights['lightgbm'] +
    preds_xgb * weights['xgboost'] +
    preds_ridge * weights['ridge']
)
preds_blend = np.maximum(preds_blend, 0)  

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': preds_blend
})
submission.to_csv('submission_blend_fixed.csv', index=False)
print("Сабмит исправлен и сохранен!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_blend_fixed.csv',   
    message='CatBoost (0.5) + LightGBM (0.3) + XGBoost (0.1) + Ridge (0.1)',  
    competition='santander-value-prediction-challenge' 
)

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')

test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')


# удаление констант (где только одно значение)
cols_to_drop = [col for col in train.columns if train[col].nunique() <= 1]
train.drop(cols_to_drop, axis=1, inplace=True)
test.drop(cols_to_drop, axis=1, inplace=True)

# удаление дубликатов колонок
train = train.loc[:, ~train.columns.duplicated()]

# статистики по строкам
def add_row_stats(df):
    features = df.drop(['ID'], axis=1)
    if 'target' in features.columns:
        features = features.drop(['target'], axis=1)
    
    df['nz_count'] = (features != 0).sum(axis=1) # Кол-во ненулевых
    df['row_mean'] = features.mean(axis=1)
    df['row_std']  = features.std(axis=1)
    df['row_min']  = features.min(axis=1)
    df['row_max']  = features.max(axis=1)
    return df

train = add_row_stats(train)
test = add_row_stats(test)

y_train_log = np.log1p(train['target'])
X_train = train.drop(['ID', 'target'], axis=1)
X_test = test.drop(['ID'], axis=1)

In [ ]:
import lightgbm as lgb

# Обучим быструю модель для отбора топ-500 признаков
selector = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
selector.fit(X_train, y_train_log)

# берем топ-500 по важности
importance = pd.DataFrame({'feature': X_train.columns, 'imp': selector.feature_importances_})
top_features = importance.sort_values('imp', ascending=False).head(500)['feature'].tolist()

X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

LightGBM с кросс-валидацией

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_top))
test_preds_lgbm = np.zeros(len(X_test_top))

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.01, # Маленький шаг
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'n_jobs': -1,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_top, y_train_log)):
    X_tr, X_val = X_train_top.iloc[train_idx], X_train_top.iloc[val_idx]
    y_tr, y_val = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params, n_estimators=10000) # До 10к деревьев
    model.fit(X_tr, y_tr, 
              eval_set=[(X_val, y_val)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(stopping_rounds=100)]) # Остановка если не улучшается
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_lgbm += model.predict(X_test_top) / kf.n_splits
    print(f"Fold {fold+1} RMSLE: {np.sqrt(mean_squared_error(y_val, oof_preds[val_idx])):.4f}")

print(f"Overall OOF RMSLE: {np.sqrt(mean_squared_error(y_train_log, oof_preds)):.4f}")

# Сабмит
submission = pd.DataFrame({'ID': test['ID'], 'target': np.expm1(test_preds_lgbm)})
submission.to_csv('submission_lgb_improved.csv', index=False)

CatBoost на логарифмических данных

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


X_train_log_features = np.log1p(X_train_top)
X_test_log_features = np.log1p(X_test_top)


kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_log_features))
test_preds_cb = np.zeros(len(X_test_log_features))

# Параметры CatBoost
cb_params = {
    'iterations': 3000,
    'learning_rate': 0.02,
    'depth': 6,
    'l2_leaf_reg': 3,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'early_stopping_rounds': 100,
    'task_type': 'CPU', 
    'verbose': 200    
}


for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_log_features, y_train_log)):
    X_tr, X_val = X_train_log_features.iloc[train_idx], X_train_log_features.iloc[val_idx]
    y_tr, y_val = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
    
    model = CatBoostRegressor(**cb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_cb += model.predict(X_test_log_features) / kf.n_splits
    
    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"Fold {fold+1} RMSE: {fold_rmse:.4f}")

full_rmse = np.sqrt(mean_squared_error(y_train_log, oof_preds))
print(f"\nИтоговый CV RMSE: {full_rmse:.4f}")

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': np.expm1(test_preds_cb) 
})

submission.to_csv('submission_catboost_logX.csv', index=False)
print("Сабмит CatBoost с логарифмированным X сохранен!")

Блендинг с CatBoost и LightLGBM

In [ ]:

blend_log = (0.6 * test_preds_cb) + (0.4 * test_preds_lgbm)

final_target = np.expm1(blend_log)

final_target = np.maximum(final_target, 0)

submission_blend = pd.DataFrame({
    'ID': test['ID'],
    'target': final_target
})

submission_blend.to_csv('submission_final_blend.csv', index=False)
print("Блендинг завершен! Файл submission_final_blend.csv создан.")

Восстановлении последовательности: поиске строк, которые идут друг за другом (похожие значения в определенных колонках). 

Отборе признаков через KS-тест: удалении колонок, распределение которых в train и test сильно отличается. 

Статистика по ненулевым значениям: модели будет обучаться на 100–300 самых важных признаках, плюс статистики по "активным" ячейкам.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from scipy.stats import ks_2samp
from sklearn.metrics import mean_squared_error
import optuna


train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')

test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

target = pd.to_numeric(train['target'], errors='coerce').values
y_train_log = np.log1p(target)

test_ids = test['ID']

print("Применяем KS-тест для отбора признаков")
cols = [c for c in train.columns if c not in ['ID', 'target']]
keep_cols = []
for col in cols:
    # Если распределения похожи (p-value > 0.05), оставляем колонку
    if ks_2samp(train[col], test[col])[1] > 0.05:
        keep_cols.append(col)



train = train[['ID', 'target'] + keep_cols]
test = test[['ID'] + keep_cols]

def get_magic_stats(df):
    X = df[keep_cols]
    X_nz = X.replace(0, np.nan)
    
    df['nz_mean'] = X_nz.mean(axis=1).fillna(0)
    df['nz_std']  = X_nz.std(axis=1).fillna(0)
    df['nz_min']  = X_nz.min(axis=1).fillna(0)
    df['nz_max']  = X_nz.max(axis=1).fillna(0)
    df['nz_sum']  = X_nz.sum(axis=1).fillna(0)
    df['nz_count'] = (X != 0).sum(axis=1)
    
    df[keep_cols] = np.log1p(X)
    return df

print("Генерация статистик по ненулевым значениям")
train = get_magic_stats(train)
test = get_magic_stats(test)

features = keep_cols + ['nz_mean', 'nz_std', 'nz_min', 'nz_max', 'nz_sum', 'nz_count']
X_train = train[features]
X_test = test[features]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def train_model(model_type, X, y, X_t):
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_t))
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        
        if model_type == 'lgb':
            m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.01, num_leaves=31, 
                                  feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=5, verbose=-1)
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        else:
            m = CatBoostRegressor(iterations=3000, learning_rate=0.02, depth=5, 
                                  random_seed=42, verbose=0, early_stopping_rounds=100)
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            
        oof[val_idx] = m.predict(X_val)
        preds += m.predict(X_t) / 5
        print(f"Fold {fold+1} finished")
    return oof, preds

print("\nОбучение LightGBM...")
oof_lgb, preds_lgb = train_model('lgb', X_train, y_train_log, X_test)

print("\nОбучение CatBoost...")
oof_cat, preds_cat = train_model('cat', X_train, y_train_log, X_test)
print("\nПоиск оптимальных весов через Optuna")
def objective(trial):
    w = trial.suggest_float('w', 0.1, 0.9)
    # Смешиваем Out-of-fold предсказания
    blend = w * oof_lgb + (1 - w) * oof_cat
    return np.sqrt(mean_squared_error(y_train_log, blend))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

best_w = study.best_params['w']
print(f"Лучший вес LGBM: {best_w:.4f}, CatBoost: {1-best_w:.4f}")

final_log_preds = best_w * preds_lgb + (1 - best_w) * preds_cat
final_target = np.expm1(final_log_preds)
# Обрезаем отрицательные 
final_target = np.clip(final_target, 0, None)

submission = pd.DataFrame({'ID': test_ids, 'target': final_target})
submission.to_csv('submission_ks_nz_optuna.csv', index=False)
print("\nГотово! Результат сохранен в submission_magic_v1.csv")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_ks_nz_optuna.csv',   
    message='KS + NZ + Optuna',  
    competition='santander-value-prediction-challenge' 
)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.decomposition import TruncatedSVD
from sklearn.random_projection import SparseRandomProjection
from sklearn.metrics import mean_squared_error
from scipy.stats import rankdata

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

y_train_log = np.log1p(train['target'])
test_ids = test['ID']

cols_to_drop = [col for col in train.columns if train[col].nunique() <= 1]
train.drop(cols_to_drop + ['ID', 'target'], axis=1, inplace=True)
test.drop(cols_to_drop + ['ID'], axis=1, inplace=True)

print("Генерация проекций SVD и SRP...")
full_data = pd.concat([train, test])

svd = TruncatedSVD(n_components=25, random_state=42)
svd_results = svd.fit_transform(full_data)

srp = SparseRandomProjection(n_components=25, dense_output=True, random_state=42)
srp_results = srp.fit_transform(full_data)

X_nz = full_data.replace(0, np.nan)
full_data['nz_mean'] = X_nz.mean(axis=1).fillna(0)
full_data['nz_std']  = X_nz.std(axis=1).fillna(0)
full_data['nz_count'] = (full_data != 0).sum(axis=1)

for i in range(25):
    full_data[f'svd_{i}'] = svd_results[:, i]
    full_data[f'srp_{i}'] = srp_results[:, i]

full_data = np.log1p(full_data)

X_train = full_data.iloc[:len(train)]
X_test = full_data.iloc[len(train):]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(model_name, X, y, X_t):
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_t))
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        if model_name == 'lgb':
            # В LGBM early_stopping теперь через callbacks
            m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.01, num_leaves=31, 
                                  subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1)
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            
        elif model_name == 'xgb':
            m = xgb.XGBRegressor(n_estimators=5000, learning_rate=0.01, max_depth=6, 
                                 subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                                 early_stopping_rounds=100, eval_metric='rmse')
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            
        elif model_name == 'cat':
            m = CatBoostRegressor(iterations=3000, learning_rate=0.02, depth=5, 
                                  random_seed=42, verbose=0, early_stopping_rounds=100)
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            
        oof[val_idx] = m.predict(X_val)
        preds += m.predict(X_t) / 5
        print(f"{model_name.upper()} Fold {fold+1} finished")
        
    return oof, preds

print("\nОбучение моделей...")
oof_lgb, preds_lgb = run_cv('lgb', X_train, y_train_log, X_test)
oof_xgb, preds_xgb = run_cv('xgb', X_train, y_train_log, X_test)
oof_cat, preds_cat = run_cv('cat', X_train, y_train_log, X_test)

print("\nПрименение Rank Blending...")
rank_lgb = rankdata(preds_lgb)
rank_xgb = rankdata(preds_xgb)
rank_cat = rankdata(preds_cat)

avg_rank = (rank_lgb + rank_xgb + rank_cat) / 3

final_target_log = np.sort(preds_lgb)[(np.argsort(np.argsort(avg_rank)))]

submission = pd.DataFrame({
    'ID': test_ids,
    'target': np.expm1(final_target_log)
})
submission['target'] = submission['target'].clip(lower=0)

submission.to_csv('submission_without_ks.csv', index=False)
print("\nГотово! Файл 'submission_without_ks.csv' готов к отправке.")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_without_ks.csv',   
    message='KS + NZ + Optuna',  
    competition='santander-value-prediction-challenge' 
)

Santander — это куски временных рядов. Значения, которые мы пытаемся предсказать, уже лежат в строке признаков, просто они там в виде прошлых значений того же клиента

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')

target_values = set(train['target'].unique())
if 0 in target_values:
    target_values.remove(0)

print(f"Уникальных ненулевых значений таргета: {len(target_values)}")

features = [col for col in train.columns if col not in ['ID', 'target']]

leak_discovery = []

for col in features:
    col_values = set(train[col].unique())
    common = col_values.intersection(target_values)
    
    if len(common) > 0:
        leak_discovery.append({
            'column': col,
            'matches_count': len(common),
            'percentage': len(common) / len(target_values) * 100
        })

leak_df = pd.DataFrame(leak_discovery).sort_values(by='matches_count', ascending=False)

print("\nКолонки, в которых чаще всего встречаются значения таргета:")
print(leak_df.head(20))

sample = train[train['target'] > 0].iloc[0]
val_target = sample['target']
matches = [col for col in features if sample[col] == val_target]

print(f"\nПример для строки ID {sample['ID']}:")
print(f"Значение Target: {val_target}")
print(f"Колонки в этой же строке, содержащие такое же значение: {matches}")

Нужно найти соседей — строки, которые относятся к одному и тому же клиенту.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

all_data = pd.concat([train.drop('target', axis=1), test])
features = [c for c in train.columns if c not in ['ID', 'target']]

# Поиск "связок" (rows that share many feature values)
leaky_cols = ['58e2e02e6', 'f190486d6', 'eeb9cd3aa', '9fd594eec', 'fb0f5dbfe']

# Попробуем найти строки, у которых совпадают все эти 5 колонок
# Это могут быть разные записи одного и того же клиента
duplicates = all_data[all_data.duplicated(subset=leaky_cols, keep=False)]
print(f"Найдено строк с одинаковыми значениями в 'утекающих' колонках: {len(duplicates)}")

# Если несколько строк имеют одинаковые значения в ряде колонок, 
# скорее всего, это один временной ряд.
all_data['magic_group'] = all_data[leaky_cols].apply(lambda x: hash(tuple(x)), axis=1)

group_counts = all_data['magic_group'].value_counts()
print(f"Количество найденных групп (цепочек): {len(group_counts[group_counts > 1])}")

train['magic_group'] = train[leaky_cols].apply(lambda x: hash(tuple(x)), axis=1)
group_targets = train.groupby('magic_group')['target'].mean()

train['group_target_mean'] = train['magic_group'].map(group_targets)
test['magic_group'] = test[leaky_cols].apply(lambda x: hash(tuple(x)), axis=1)
test['group_target_mean'] = test['magic_group'].map(group_targets)

Данные - это не 50 000 уникальных клиентов, это около 900–1000 реальных объектов, чья история была разрезана на мелкие кусочки и перемешана.
Утечка подтверждена. Те 881 группа (группы цепочек) - это сообщества строк, которые принадлежат одному и тому же временному ряду

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

leaky_cols = ['58e2e02e6', 'f190486d6', 'eeb9cd3aa', '9fd594eec', 'fb0f5dbfe', 
              '6eef030c1', '15ace8c9f', '20aa07010', '58e056e12', '1702b5bf0']

print("Группировка строк")
all_df_leaky = pd.concat([train[leaky_cols], test[leaky_cols]], axis=0)

group_keys = all_df_leaky.apply(lambda x: "_".join(x.map(str)), axis=1)

train['group_key'] = group_keys.iloc[:len(train)].values
test['group_key'] = group_keys.iloc[len(train):].values

group_stats = train.groupby('group_key')['target'].agg(['mean', 'std', 'count']).reset_index()
group_stats.columns = ['group_key', 'group_target_mean', 'group_target_std', 'group_count']

train = train.merge(group_stats, on='group_key', how='left')
test = test.merge(group_stats, on='group_key', how='left')

train['group_target_mean'] = train['group_target_mean'].fillna(0)
train['group_target_std']  = train['group_target_std'].fillna(0)
test['group_target_mean']  = test['group_target_mean'].fillna(0)
test['group_target_std']   = test['group_target_std'].fillna(0)

train['leaky_median'] = train[leaky_cols].median(axis=1)
test['leaky_median'] = test[leaky_cols].median(axis=1)

train.drop('group_key', axis=1, inplace=True)
test.drop('group_key', axis=1, inplace=True)

print("Признаки созданы успешно!")
print(f"Количество строк в тесте, которые 'узнали' свой таргет из трейна: {test[test['group_count'] > 1].shape[0]}")

Обучим ещё раз CatBoost и LightGBM, но уже на новых данных: 
LGBMRegressor, кросс-валидация KFold (5 фолдов), Early Stopping, логарифмирование таргета (log1p). 

Это "чистый" бустинг на отобранных признаках. Здесь мы еще не создаем новые фичи, а настраиваем модель: длинное обучение (10к деревьев) с маленьким шагом (0.01). 

Вывод: vодель дает неплохую базу, но упирается в предел из-за огромного количества нулей в данных, которые бустинг воспринимает как обычные значения. 

Результаты такие: 1.39895 и 1.44623. 




In [ ]:
import pandas as pd
import numpy as np

leaky_cols = ['58e2e02e6', 'f190486d6', 'eeb9cd3aa', '9fd594eec', 'fb0f5dbfe', 
              '6eef030c1', '15ace8c9f', '20aa07010', '58e056e12', '1702b5bf0']

train_leaky = train[leaky_cols].apply(lambda x: "_".join(x.map(str)), axis=1)
test_leaky = test[leaky_cols].apply(lambda x: "_".join(x.map(str)), axis=1)

train['group_key'] = train_leaky
test['group_key'] = test_leaky

group_stats = train.groupby('group_key')['target'].agg(['mean', 'std', 'count']).reset_index()
group_stats.columns = ['group_key', 'group_target_mean', 'group_target_std', 'group_count']

cols_to_drop = ['group_target_mean', 'group_target_std', 'group_count']
train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
test = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

train = train.merge(group_stats, on='group_key', how='left')
test = test.merge(group_stats, on='group_key', how='left')

train['leaky_median'] = train[leaky_cols].median(axis=1)
test['leaky_median'] = test[leaky_cols].median(axis=1)

magic_feats = ['group_target_mean', 'group_target_std', 'group_count', 'leaky_median']
for col in magic_feats:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

for df in [train, test]:
    df['group_target_mean'] = np.log1p(df['group_target_mean'])
    df['group_target_std'] = np.log1p(df['group_target_std'])
    df['leaky_median'] = np.log1p(df['leaky_median'])

final_features = [c for c in top_features if c not in ['ID', 'target', 'group_key']] + magic_feats

X_train = train[final_features]
X_test = test[final_features]

print(f"Успех! Теперь в X_train {X_train.shape[1]} признаков.")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_top))
test_preds_lgbm = np.zeros(len(X_test_top))

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.01, # Маленький шаг
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'n_jobs': -1,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_top, y_train_log)):
    X_tr, X_val = X_train_top.iloc[train_idx], X_train_top.iloc[val_idx]
    y_tr, y_val = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params, n_estimators=10000) # До 10к деревьев
    model.fit(X_tr, y_tr, 
              eval_set=[(X_val, y_val)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(stopping_rounds=100)]) # Остановка если не улучшается
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_lgbm += model.predict(X_test_top) / kf.n_splits
    print(f"Fold {fold+1} RMSLE: {np.sqrt(mean_squared_error(y_val, oof_preds[val_idx])):.4f}")

print(f"Overall OOF RMSLE: {np.sqrt(mean_squared_error(y_train_log, oof_preds)):.4f}")

# Сабмит
submission = pd.DataFrame({'ID': test['ID'], 'target': np.expm1(test_preds_lgbm)})
submission.to_csv('submission_lgb_improved_new_features.csv', index=False)

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_lgb_improved_new_features.csv',   
    message='LightGBM, new features',  
    competition='santander-value-prediction-challenge' 
)

CatBoost с логарифмированием признаков 

CatBoostRegressor, Log-transform X (логарифмирование входных признаков) 

Применение log1p не только к таргету, но и к самим признакам. Это "сжимает" выбросы в данных Santander, делая распределения более колоколообразными. 

Вывод: CatBoost нативно лучше справляется с разреженными sparse данными. Логарифмирование фич помогло модели быстрее находить закономерности. 

Результаты: 1.39706 и 1.43956

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


X_train_log_features = np.log1p(X_train_top)
X_test_log_features = np.log1p(X_test_top)


kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_log_features))
test_preds_cb = np.zeros(len(X_test_log_features))

# Параметры CatBoost
cb_params = {
    'iterations': 3000,
    'learning_rate': 0.02,
    'depth': 6,
    'l2_leaf_reg': 3,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'early_stopping_rounds': 100,
    'task_type': 'CPU', 
    'verbose': 200    
}


for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_log_features, y_train_log)):
    X_tr, X_val = X_train_log_features.iloc[train_idx], X_train_log_features.iloc[val_idx]
    y_tr, y_val = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
    
    model = CatBoostRegressor(**cb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_cb += model.predict(X_test_log_features) / kf.n_splits
    
    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"Fold {fold+1} RMSE: {fold_rmse:.4f}")

full_rmse = np.sqrt(mean_squared_error(y_train_log, oof_preds))
print(f"\nИтоговый CV RMSE: {full_rmse:.4f}")

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': np.expm1(test_preds_cb) 
})

submission.to_csv('submission_catboost_logX_improved_new_features.csv', index=False)
print("Сабмит CatBoost с логарифмированным X сохранен!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_catboost_logX_improved_new_features.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

Бленлинг из CatBoost, LightGBM, 
 Переход от одиночных моделей к блендингу. Введены первые "безопасные" признаки — статистики по ненулевым значениям строки, которые позволяют модели оценить реальный масштаб клиента. Сочетание двух разных алгоритмов (CB, LGBM) дало лучший результат (Private 1.38583, public 1.44044), так как модели ошибаются в разных местах и компенсируют друг друга.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from scipy.stats import rankdata
from sklearn.metrics import mean_squared_error

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')
test_ids = test['ID']

cols_to_drop = [col for col in train.columns if train[col].nunique() <= 1]
leaky_cols = ['58e2e02e6', 'f190486d6', 'eeb9cd3aa', '9fd594eec', 'fb0f5dbfe', '6eef030c1', '15ace8c9f', '20aa07010']

y_log = np.log1p(train['target'])
X = train.drop(cols_to_drop + ['ID', 'target'], axis=1)
X_test = test.drop(cols_to_drop + ['ID'], axis=1)

def add_safe_features(df):
    X_raw = df.copy()
    X_nz = X_raw.replace(0, np.nan)
    df['nz_mean'] = X_nz.mean(axis=1).fillna(0)
    df['nz_count'] = (X_raw != 0).sum(axis=1)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = np.log1p(df[num_cols])
    return df

print("Генерация признаков...")
X = add_safe_features(X)
X_test = add_safe_features(X_test)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def get_preds(X, y, X_t):
    oof_cb = np.zeros(len(X))
    preds_cb = np.zeros(len(X_t))
    
    oof_lgb = np.zeros(len(X))
    preds_lgb = np.zeros(len(X_t))

    for fold, (tr, val) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr], X.iloc[val]
        y_tr, y_val = y.iloc[tr], y.iloc[val]
        
        # CatBoost 
        cb = CatBoostRegressor(iterations=3000, learning_rate=0.015, depth=5, 
                               l2_leaf_reg=10, random_seed=42, verbose=0)
        cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
        oof_cb[val] = cb.predict(X_val)
        preds_cb += cb.predict(X_t) / 5
        
        # LightGBM 
        lgbm = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.01, num_leaves=15, 
                                 feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1, verbose=-1)
        lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                 callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        oof_lgb[val] = lgbm.predict(X_val)
        preds_lgb += lgbm.predict(X_t) / 5
        
        print(f"Fold {fold+1} finished. OOF RMSE: {np.sqrt(mean_squared_error(y_val, oof_cb[val])):.4f}")
        
    return preds_cb, preds_lgb

p_cb, p_lgb = get_preds(X, y_log, X_test)

final_log_preds = (0.6 * p_cb) + (0.4 * p_lgb)

submission = pd.DataFrame({
    'ID': test_ids,
    'target': np.expm1(final_log_preds)
})
submission['target'] = submission['target'].clip(lower=0)

submission.to_csv('submission_cb_lgbm_pro.csv', index=False)
print("Готово!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_cb_lgbm_pro.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

Поиск временных цепочек 

Анализ пересечения значений (leak_discovery), словари Python (mapping) для поиска связей между строками.
Мы пытаемся найти строки, где "хвост" признаков одной строки совпадает с "головой" другой, чтобы восстановить порядок транзакций.
Были найдены первые совпадения. Стало понятно, что данные — это разрезанные временные ряды, но простая замена значений нашла слишком мало пар для серьезного улучшения. Результат сильно ухудшился (1.42480 и 1.48335)

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

# берем топ-40 колонок с наибольшим числом совпадений с таргетом
target_values = set(train['target'].unique())
features = [col for col in train.columns if col not in ['ID', 'target']]

leak_discovery = []
for col in features:
    common = set(train[col].unique()).intersection(target_values)
    leak_discovery.append({'column': col, 'matches': len(common)})

leak_df = pd.DataFrame(leak_discovery).sort_values(by='matches', ascending=False)
magic_cols = leak_df['column'].head(40).tolist()

print(f"Используем {len(magic_cols)} колонок для поиска цепочек.")

# Строим словарь: { "хвост_строки" : таргет }
# Хвост — это колонки с индексом 1 и далее
train_magic = train[magic_cols].values
targets = train['target'].values

mapping = {}
for i in range(len(train)):
    # кортеж из значений со сдвигом (со 2-й колонки до конца)
    suffix = tuple(train_magic[i, 1:]) 
    mapping[suffix] = targets[i]

# 4. Ищем соответствия в тесте
best_sub = pd.read_csv('submission_without_ks.csv')
final_preds = best_sub['target'].values

test_magic = test[magic_cols].values
count_replaced = 0

for i in range(len(test)):
    # Берем "голову" строки теста (с 1-й по 39-ю колонку)
    prefix = tuple(test_magic[i, :-1])
    
    # не является ли эта "голова" чьим-то "хвостом" из трейна?
    if prefix in mapping:
        final_preds[i] = mapping[prefix]
        count_replaced += 1

print(f"Заменено предсказаний в тесте: {count_replaced}")

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': final_preds
})
submission.to_csv('submission_magic_leak_final.csv', index=False)
print("Файл submission_magic_leak_final.csv готов!")

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_magic_leak_final.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

"Золотой список" из 40 колонок, поиск цепочек по всему датасету (Train + Test). 
 В отличие от предыдущего кода, здесь используется строгий порядок 40 специфических колонок. Мы ищем таргет не в самой строке, а в первой колонке "будущей" строки в цепочке/ 
 Количество найденных связей выросло. Этот подход подтвердил гипотезу о структуре данных, хотя прямое замещение всё еще затронуло лишь малую часть теста. Результат: 1.39254, 1.44780, private улучшился.


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

magic_cols_raw = [
    'f190486d6', '58e2e02e6', 'eeb9cd3aa', '9fd594eec', 'fb0f5dbfe', 
    '6eef030c1', '15ace8c9f', '20aa07010', '58e056e12', '1702b5bf0', 
    'b43a7cfd5', 'd6bb78916', '024c577b9', '58232a6fb', '241f0f867', 
    '2ec5b290f', '324921c7b', '62e59a501', 'f74e8f13d', 'fb49e4212',
    'b761b8e0e', 'd1d1cafe3', '03490ef8c', 'c380056bb', '371ff7a11',
    '8d401b32e', '3e3ea106e', 'b94360a3b', 'a4bbe86e5', '6cf7866c1',
    '4cffe31c7', '95de610db', '7bddf55e1', 'a8b590c6e', 'ce30bda90',
    '78cbd925b', '844df03d7', '7ddac276f', 'b3a30c6a2', 'fb5e1b2b7'
]
magic_cols = [c for c in magic_cols_raw if c in train.columns]

# (Private 1.38583 / Public 1.44044)
sub = pd.read_csv('/kaggle/working/submission_cb_lgbm_pro.csv')
final_preds = sub['target'].values

all_data = pd.concat([train[magic_cols], test[magic_cols]], axis=0).values

mapping = {}
for i in range(len(all_data)):
    key = tuple(all_data[i, 1:])
    mapping[key] = all_data[i, 0]

test_magic_values = test[magic_cols].values
count_replaced = 0

for i in range(len(test)):
    query = tuple(test_magic_values[i, :-1])
    
    if query in mapping:
        val = mapping[query]
        if val > 0: 
            final_preds[i] = val
            count_replaced += 1

print(f"Найдено и заменено связей: {count_replaced}")

sub['target'] = final_preds
sub.to_csv('submission_real_magic_leak.csv', index=False)

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_real_magic_leak.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

LightGBM с параметром importance_type='gain', удаление константных признаков. 

Здесь мы убираем "шум". Из 5000 признаков мы оставляем только топ-500, которые реально вносят вклад в снижение ошибки. 

Очистка данных позволила моделям не переобучаться на случайных совпадениях в 4500 бесполезных колонках. 

Результат хуже: 1.43735 и 1.48336

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

def find_top_features(train_df, n_top=500):
    print("Процесс отбора признаков...")
    X = train_df.drop(['ID', 'target'], axis=1, errors='ignore')
    y = np.log1p(train_df['target'])
    
    initial_cols = X.columns.tolist()
    X = X.loc[:, X.nunique() > 1]
    print(f"Удалено {len(initial_cols) - X.shape[1]} константных колонок.")
    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        importance_type='gain', # Оценка по вкладу в снижение ошибки
        random_state=42,
        verbose=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50)]
    )
    
    # 4. Извлечение важности
    importance_df = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values(by='importance', ascending=False)
    
    top_features = importance_df.head(n_top)['feature'].tolist()
    
    print(f"Отбор завершен. Выбрано топ-{n_top} признаков.")
    return top_features, importance_df

top_features, full_importance = find_top_features(train, n_top=500)

print("\nСамые важные колонки:")
print(full_importance.head(10))

Интеграция цепочек в обучение (magic_sum, magic_mean, magic_first), CatBoost с низким шагом (0.005). 

Мы перестали вручную менять ответы. Теперь информация о временных связях подается модели в виде новых признаков. 

Это заставило модель саму «выучить» структуру утечки. Медленное обучение на 6000 итераций позволило выжать максимум из этих связей. 

Результат: 1.43735 и 1.48336

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')

magic_cols_raw = [
    'f190486d6', '58e2e02e6', 'eeb9cd3aa', '9fd594eec', 'fb0f5dbfe', 
    '6eef030c1', '15ace8c9f', '20aa07010', '58e056e12', '1702b5bf0', 
    'b43a7cfd5', 'd6bb78916', '024c577b9', '58232a6fb', '241f0f867', 
    '2ec5b290f', '324921c7b', '62e59a501', 'f74e8f13d', 'fb49e4212',
    'b761b8e0e', 'd1d1cafe3', '03490ef8c', 'c380056bb', '371ff7a11',
    '8d401b32e', '3e3ea106e', 'b94360a3b', 'a4bbe86e5', '6cf7866c1',
    '4cffe31c7', '95de610db', '7bddf55e1', 'a8b590c6e', 'ce30bda90',
    '78cbd925b', '844df03d7', '7ddac276f', 'b3a30c6a2', 'fb5e1b2b7'
]
magic_cols = [c for c in magic_cols_raw if c in train.columns]

def create_leaky_features(df):
    subset = df[magic_cols]
    df['magic_sum'] = subset.sum(axis=1)
    df['magic_mean'] = subset.mean(axis=1)
    df['magic_std'] = subset.std(axis=1)
    df['magic_nonzero_count'] = (subset != 0).sum(axis=1)
    
    df['magic_first'] = subset[magic_cols[0]]
    
    for col in magic_cols + ['magic_sum', 'magic_mean', 'magic_first']:
        df[col] = np.log1p(df[col])
    return df

train = create_leaky_features(train)
test = create_leaky_features(test)

y_train_log = np.log1p(train['target'])
features = list(set(top_features + magic_cols + ['magic_sum', 'magic_mean', 'magic_std', 'magic_nonzero_count', 'magic_first']))
features = [c for c in features if c in train.columns]

X_train = train[features]
X_test = test[features]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def get_oof_preds(X, y, X_t):
    oof_preds = np.zeros(len(X))
    sub_preds = np.zeros(len(X_t))
    
    for fold, (tr, val) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr], X.iloc[val]
        y_tr, y_val = y.iloc[tr], y.iloc[val]
        
        model = CatBoostRegressor(
            iterations=6000, 
            learning_rate=0.005, 
            depth=5, 
            l2_leaf_reg=15, 
            random_seed=42, 
            verbose=0,
            early_stopping_rounds=200
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        
        oof_preds[val] = model.predict(X_val)
        sub_preds += model.predict(X_t) / 5
        print(f"Fold {fold+1} finished. RMSE: {np.sqrt(mean_squared_error(y_val, oof_preds[val])):.4f}")
        
    return sub_preds

print("Запуск обучения ")
final_log_preds = get_oof_preds(X_train, y_train_log, X_test)

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': np.expm1(final_log_preds)
})
submission['target'] = submission['target'].clip(lower=0)
submission.to_csv('submission_magic_features_only.csv', index=False)

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_magic_features_only.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

Пока лучшее, что есть. 
Агрегаты и Тройной Ансамбль 

nz_median (медиана ненулевых), RandomForestRegressor, XGBoost, взвешенный бленд по трем моделям. 

Самый сильный табличный этап. Мы добавили Random Forest, который за счет бэггинга стабилизировал результат на шумных данных. Признак nz_median стал самым важным в проекте. 

Достигнут результат 1.36199 и 1.40349. Комбинация бустинг + лес + статистики — это самый стабильный путь.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')
y_log = np.log1p(train['target'])

def get_advanced_aggregates(df):
    cols = [c for c in df.columns if c not in ['ID', 'target']]
    X_raw = df[cols]
    
    X_nz = X_raw.replace(0, np.nan)
    
    df['nz_count'] = (X_raw != 0).sum(axis=1)
    df['nz_sum']   = X_raw.sum(axis=1)
    df['nz_mean']  = X_nz.mean(axis=1).fillna(0)
    df['nz_median']= X_nz.median(axis=1).fillna(0) 
    df['nz_std']   = X_nz.std(axis=1).fillna(0)
    df['nz_min']   = X_nz.min(axis=1).fillna(0)
    df['nz_max']   = X_nz.max(axis=1).fillna(0)
    df['nz_fraction'] = df['nz_count'] / len(cols)
    df['nz_avg_size'] = df['nz_sum'] / (df['nz_count'] + 1e-9)
    
    for col in ['nz_sum', 'nz_mean', 'nz_median', 'nz_min', 'nz_max', 'nz_avg_size']:
        df[col] = np.log1p(df[col])
        
    df[cols] = np.log1p(X_raw)
    return df

print("Генерация расширенных признаков...")
train = get_advanced_aggregates(train)
test = get_advanced_aggregates(test)

# отбор признаков
X_all = train.drop(['ID', 'target'], axis=1)
selector = lgb.LGBMRegressor(n_estimators=500, random_state=42, verbose=-1)
selector.fit(X_all, y_log)
top_features = pd.DataFrame({'f': X_all.columns, 'i': selector.feature_importances_}).sort_values('i', ascending=False).head(1000)['f'].tolist()

X_train = train[top_features]
X_test = test[top_features]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def train_suite(X, y, X_t):
    preds_lgb = np.zeros(len(X_t))
    preds_rf  = np.zeros(len(X_t))
    preds_xgb = np.zeros(len(X_t))
    
    for fold, (tr, val) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr], X.iloc[val]
        y_tr, y_val = y.iloc[tr], y.iloc[val]
        
        # LightGBM
        m_lgb = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.01, num_leaves=31, verbose=-1)
        m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100)])
        preds_lgb += m_lgb.predict(X_t) / 5
        
        # RandomForest 
        m_rf = RandomForestRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)
        m_rf.fit(X_tr, y_tr)
        preds_rf += m_rf.predict(X_t) / 5

        # XGBoost
        m_xgb = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6, n_jobs=-1)
        m_xgb.fit(X_tr, y_tr)
        preds_xgb += m_xgb.predict(X_t) / 5
        
        print(f"Fold {fold+1} finished.")
        
    return preds_lgb, preds_rf, preds_xgb

p_lgb, p_rf, p_xgb = train_suite(X_train, y_log, X_test)

final_preds_log = (0.45 * p_lgb) + (0.40 * p_rf) + (0.15 * p_xgb)

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': np.expm1(final_preds_log)
})
submission.to_csv('submission_blending_again.csv', index=False)

In [ ]:
from kaggle import api

api.competition_submit(
    file_name='submission_blending_again.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

Pseudo-labeling (самообучение), TruncatedSVD (проекции), оптимизация памяти (float32), max_features='sqrt' для ускорения. 

Использование тестовой выборки для обучения. Мы берем лучшие предсказания теста, называем их "правдой" и учим модель на 53 000 строк вместо 4 000. Дополнительно добавлен SVD для поиска скрытых связей в разреженной матрице. 

Самый сложный и тяжелый пайплайн. Позволил модели увидеть глобальную структуру данных всего соревнования. Оптимизированная версия кода ускорила выполнение. 
Результат ухудшился: 1.40245 и 1.44913

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import TruncatedSVD

print("Загрузка данных")
train = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/santander-value-prediction-challenge/test.csv')
best_preds_sub = pd.read_csv('/kaggle/working/submission_blending_again.csv')

def downcast_df(df):
    float_cols = [c for c in df.columns if df[c].dtype == "float64"]
    int_cols = [c for c in df.columns if df[c].dtype == "int64"]
    df[float_cols] = df[float_cols].astype(np.float32)
    df[int_cols] = df[int_cols].astype(np.int32)
    return df

train = downcast_df(train)
test = downcast_df(test)

y_train_log = np.log1p(train['target']).astype(np.float32)
y_test_pseudo_log = np.log1p(best_preds_sub['target']).astype(np.float32)

print("Отбор признаков")
X_full = train.drop(['ID', 'target'], axis=1)
constant_cols = [col for col in X_full.columns if X_full[col].nunique() <= 1]
X_full.drop(constant_cols, axis=1, inplace=True)

selector = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1, n_jobs=-1)
selector.fit(X_full, y_train_log)
top_cols = pd.Series(selector.feature_importances_, index=X_full.columns).sort_values(ascending=False).head(1000).index.tolist()

def get_final_features_optimized(df_train, df_test, cols_to_use):
    full_data = pd.concat([df_train[cols_to_use], df_test[cols_to_use]])
    
    X_raw = full_data.values
    full_data['nz_count'] = (X_raw != 0).sum(axis=1)
    full_data['nz_mean'] = np.true_divide(X_raw.sum(axis=1), (X_raw != 0).sum(axis=1) + 1e-9)
    

    print("Генерация SVD")
    svd = TruncatedSVD(n_components=20, random_state=42)
    svd_feats = svd.fit_transform(full_data[cols_to_use])
    for i in range(20):
        full_data[f'svd_{i}'] = svd_feats[:, i].astype(np.float32)
    
    full_data[cols_to_use] = np.log1p(full_data[cols_to_use])
    
    return full_data.iloc[:len(df_train)], full_data.iloc[len(df_train):]

X_train, X_test = get_final_features_optimized(train, test, top_cols)

X_combined = pd.concat([X_train, X_test], axis=0)
y_combined = np.concatenate([y_train_log, y_test_pseudo_log])

def train_final_ensemble_fast(X_comb, y_comb, X_target_test):
    print("Обучение LightGBM")
    m_lgb = lgb.LGBMRegressor(n_estimators=3000, 
                              learning_rate=0.015, 
                              num_leaves=31, 
                              feature_fraction=0.6, 
                              bagging_fraction=0.7, 
                              n_jobs=-1, 
                              verbose=-1)
    m_lgb.fit(X_comb, y_comb)
    p_lgb = m_lgb.predict(X_target_test)
    
    print("Обучение RandomForest ")
    m_rf = RandomForestRegressor(n_estimators=150, # Уменьшим до 150, разница в качестве будет < 0.001
                                 max_depth=15,    
                                 max_features='sqrt', 
                                 n_jobs=-1, 
                                 random_state=42)
    m_rf.fit(X_comb, y_comb)
    p_rf = m_rf.predict(X_target_test)
    
    return (0.5 * p_lgb) + (0.5 * p_rf)

print("Начало Pseudo-labeling")
final_preds_log = train_final_ensemble_fast(X_combined, y_combined, X_test)

submission = pd.DataFrame({
    'ID': test['ID'],
    'target': np.expm1(final_preds_log)
})
submission['target'] = submission['target'].clip(lower=0)
submission.to_csv('submission_pseudo_svd_final.csv', index=False)
print("Готово!")

Загрузка данных
Отбор признаков


/tmp/ipykernel_58/3940638836.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_data['nz_count'] = (X_raw != 0).sum(axis=1)
/tmp/ipykernel_58/3940638836.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_data['nz_mean'] = np.true_divide(X_raw.sum(axis=1), (X_raw != 0).sum(axis=1) + 1e-9)


Генерация SVD


/tmp/ipykernel_58/3940638836.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_data[f'svd_{i}'] = svd_feats[:, i].astype(np.float32)
/tmp/ipykernel_58/3940638836.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_data[f'svd_{i}'] = svd_feats[:, i].astype(np.float32)
/tmp/ipykernel_58/3940638836.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.

Начало Pseudo-labeling
Обучение LightGBM
Обучение RandomForest 
Готово!


In [5]:
from kaggle import api

api.competition_submit(
    file_name='submission_pseudo_svd_final.csv',   
    message='CatBoost, new features',  
    competition='santander-value-prediction-challenge' 
)

100%|██████████| 1.34M/1.34M [00:00<00:00, 6.19MB/s]


{"message": "Successfully submitted to Santander Value Prediction Challenge", "ref": 55643751}